# Assignment 1 Solution: Building a Rival-Company NLP Corpus

## Learning Objectives
By working through this notebook you will be able to:

1. **Construct** a multi-year SEC 10-K corpus for two rival public companies.
2. **Vectorize** text using TF-IDF and interpret top distinguishing terms.
3. **Measure** document similarity with cosine similarity across fiscal years.
4. **Train** a lightweight PyTorch Bag-of-Words classifier to predict company from text.
5. **Interpret** model performance in a business context.

## Companies Used in This Solution
| Field | Company A | Company B |
|---|---|---|
| Name | NVIDIA Corporation | Advanced Micro Devices (AMD) |
| Ticker | NVDA | AMD |
| CIK | 0001045810 | 0000002488 |
| NAICS | 3344 | 3344 |
| Sector | Semiconductors | Semiconductors |

> **Good Programming Practice:** Save intermediate artifacts (CSV, JSON, model weights)
> after each expensive computation so you can reload them without re-running everything.

In [ ]:
# ── Cell 1: Install / import dependencies ─────────────────────────────────────
# Run this cell first on Google Colab. Local installs: `uv pip install ...`
import subprocess, sys

required = [
    "requests", "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "torch",
]
for pkg in required:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, re, json, time, requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print("Dependencies loaded ✓")

In [ ]:
# ── Cell 2: Configuration & artifact directory ─────────────────────────────────
ARTIFACTS = "./M01_A_sol_artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)

COMPANIES = {
    "NVDA": {"name": "NVIDIA Corporation",         "cik": "0001045810"},
    "AMD":  {"name": "Advanced Micro Devices Inc.", "cik": "0000002488"},
}
YEARS = [2020, 2021, 2022, 2023, 2024]
HEADERS = {"User-Agent": "AD698-Student research@bu.edu"}

print(f"Artifacts will be saved to: {os.path.abspath(ARTIFACTS)}")

## Part 1 — Market Context via Yahoo Finance

Before diving into text, ground the analysis with market data.
The table below is built from Yahoo Finance price/fundamental data.
We use `yfinance` for convenience; you could also use the free
[Yahoo Finance API](https://finance.yahoo.com).

In [ ]:
# ── Cell 3: Yahoo Finance market-context table ─────────────────────────────────
try:
    import yfinance as yf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
    import yfinance as yf

rows = []
for ticker, meta in COMPANIES.items():
    info = yf.Ticker(ticker).info
    rows.append({
        "Ticker": ticker,
        "Company": meta["name"],
        "Market Cap ($B)": round(info.get("marketCap", 0) / 1e9, 1),
        "Revenue TTM ($B)": round(info.get("totalRevenue", 0) / 1e9, 1),
        "Gross Margin %": round(info.get("grossMargins", 0) * 100, 1),
        "P/E Ratio": round(info.get("trailingPE", float("nan")), 1),
        "52w High": round(info.get("fiftyTwoWeekHigh", 0), 2),
        "52w Low":  round(info.get("fiftyTwoWeekLow", 0), 2),
    })

market_df = pd.DataFrame(rows)
market_df.to_csv(f"{ARTIFACTS}/company_context.csv", index=False)
print("Saved company_context.csv")
market_df

## Part 2 — SEC EDGAR 10-K Download

We pull Item 1A (Risk Factors) and Item 7 (MD&A) from annual 10-K filings
using the SEC EDGAR full-text search API.

**Why these sections?**
- **Item 1A** describes risks specific to the company's business model and
  competitive environment — rich signal about strategy differences.
- **Item 7** contains management's narrative about financial results —
  useful for year-over-year trend analysis.

The SEC limits requests to **10 per second**; we add a small `time.sleep`.

In [ ]:
# ── Cell 4: Fetch 10-K filings from SEC EDGAR ─────────────────────────────────
def get_10k_text(cik: str, year: int, headers: dict) -> str:
    """Return concatenated Item 1A + Item 7 text for a given company/year."""
    cik_padded = cik.lstrip("0").zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    filings = requests.get(url, headers=headers).json()
    recent = filings.get("filings", {}).get("recent", {})
    forms  = recent.get("form", [])
    dates  = recent.get("filingDate", [])
    accs   = recent.get("accessionNumber", [])

    # Find the 10-K filed closest to the fiscal year end
    target_acc = None
    for form, date, acc in zip(forms, dates, accs):
        if form == "10-K" and str(year) in date:
            target_acc = acc.replace("-", "")
            break

    if not target_acc:
        return ""

    idx_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{target_acc}/{target_acc}-index.json"
    try:
        idx = requests.get(idx_url, headers=headers).json()
    except Exception:
        return ""

    # Find primary document
    doc_name = None
    for item in idx.get("directory", {}).get("item", []):
        name = item.get("name", "")
        if name.endswith(".htm") and "10k" in name.lower():
            doc_name = name
            break
    if not doc_name:
        for item in idx.get("directory", {}).get("item", []):
            name = item.get("name", "")
            if name.endswith(".htm"):
                doc_name = name
                break

    if not doc_name:
        return ""

    doc_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{target_acc}/{doc_name}"
    raw = requests.get(doc_url, headers=headers).text

    # Strip HTML tags
    clean = re.sub(r"<[^>]+>", " ", raw)
    clean = re.sub(r"\s+", " ", clean).strip()
    return clean[:150_000]   # cap at ~150k chars per filing

records = []
for ticker, meta in COMPANIES.items():
    for year in YEARS:
        print(f"Fetching {ticker} {year} ...", end=" ")
        text = get_10k_text(meta["cik"], year, HEADERS)
        records.append({"ticker": ticker, "year": year, "text": text})
        print(f"{len(text):,} chars")
        time.sleep(0.15)   # respect SEC rate limit

corpus_df = pd.DataFrame(records)
corpus_df.to_csv(f"{ARTIFACTS}/corpus.csv", index=False)
print("\nSaved corpus.csv")
corpus_df.head()

## Part 3 — Chunking the Corpus

Long documents are split into **512-token chunks** (≈ 400 words) so that:
- Each chunk is an independent training sample for the classifier.
- TF-IDF vectors represent focused passages rather than entire filings.

> **Good Practice:** Save `chunks.csv` — subsequent assignments (A2, A3, A4)
> load this file directly so you never need to re-download from EDGAR.

In [ ]:
# ── Cell 5: Chunk documents into ~400-word passages ───────────────────────────
CHUNK_SIZE = 400  # words

def chunk_text(text: str, ticker: str, year: int, size: int = CHUNK_SIZE):
    words  = text.split()
    chunks = [" ".join(words[i:i+size]) for i in range(0, len(words), size) if words[i:i+size]]
    return [{"ticker": ticker, "year": year, "chunk_id": j, "text": c}
            for j, c in enumerate(chunks)]

all_chunks = []
for _, row in corpus_df.iterrows():
    all_chunks.extend(chunk_text(row["text"], row["ticker"], row["year"]))

chunks_df = pd.DataFrame(all_chunks)
chunks_df.to_csv(f"{ARTIFACTS}/chunks.csv", index=False)
print(f"Total chunks: {len(chunks_df):,}")
chunks_df["ticker"].value_counts()

## Part 4 — TF-IDF Vectorization

### Concept: Term Frequency–Inverse Document Frequency

$$\text{TF-IDF}(t, d) = \underbrace{\frac{\text{count}(t,d)}{\text{len}(d)}}_{\text{TF}} \times \underbrace{\log\frac{N}{df(t)}}_{\text{IDF}}$$

- **TF** rewards terms that appear often *in this document*.
- **IDF** penalizes terms that appear in *every* document (e.g., "the", "company").
- The product highlights terms that are *distinctive* to a specific document.

We use **bigrams** (`ngram_range=(1,2)`) because phrases like "data center" or
"supply chain" are more informative than individual words.

In [ ]:
# ── Cell 6: Fit TF-IDF vectorizer ─────────────────────────────────────────────
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=3,
)
tfidf_matrix = vectorizer.fit_transform(chunks_df["text"])
vocab = vectorizer.get_feature_names_out().tolist()

# Save vocab
with open(f"{ARTIFACTS}/vocab.json", "w") as f:
    json.dump(vocab, f)

# Save TF-IDF matrix as dense CSV (first 1000 cols for manageability)
tfidf_df = pd.DataFrame(
    tfidf_matrix[:, :1000].toarray(),
    columns=vocab[:1000]
)
tfidf_df.to_csv(f"{ARTIFACTS}/tfidf_matrix.csv", index=False)
print(f"TF-IDF matrix: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(vocab):,}")
print(f"Saved vocab.json and tfidf_matrix.csv")

In [ ]:
# ── Cell 7: Top-20 distinctive terms per company (seaborn) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (ticker, _) in zip(axes, COMPANIES.items()):
    mask = chunks_df["ticker"] == ticker
    mean_tfidf = np.asarray(tfidf_matrix[mask].mean(axis=0)).flatten()
    top_idx   = mean_tfidf.argsort()[-20:][::-1]
    top_terms = [vocab[i] for i in top_idx]
    top_vals  = mean_tfidf[top_idx]

    sns.barplot(x=top_vals, y=top_terms, ax=ax, palette="Blues_d")
    ax.set_title(f"Top-20 TF-IDF Terms — {ticker}", fontsize=13)
    ax.set_xlabel("Mean TF-IDF Score")
    ax.set_ylabel("")

plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/top_tfidf_terms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved top_tfidf_terms.png")

## Part 5 — Cosine Similarity Over Time

Cosine similarity measures how aligned two document vectors are in TF-IDF space:

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$$

- **1.0** → documents are identical in term distribution.
- **0.0** → documents share no terms.

We compute the average cosine similarity between all NVDA chunks and all AMD
chunks for each fiscal year to detect language convergence/divergence.

In [ ]:
# ── Cell 8: Year-by-year cosine similarity ────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

sim_rows = []
for year in YEARS:
    nvda_mask = (chunks_df["ticker"] == "NVDA") & (chunks_df["year"] == year)
    amd_mask  = (chunks_df["ticker"] == "AMD")  & (chunks_df["year"] == year)
    if nvda_mask.sum() == 0 or amd_mask.sum() == 0:
        continue
    mat_nvda = tfidf_matrix[nvda_mask]
    mat_amd  = tfidf_matrix[amd_mask]
    sim = cosine_similarity(mat_nvda, mat_amd).mean()
    sim_rows.append({"year": year, "mean_cosine_sim": sim})

sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(f"{ARTIFACTS}/cosine_sim_by_year.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=sim_df, x="year", y="mean_cosine_sim",
             marker="o", linewidth=2.5, ax=ax)
ax.set_title("NVDA vs AMD — Mean Cosine Similarity by Fiscal Year")
ax.set_xlabel("Fiscal Year"); ax.set_ylabel("Mean Cosine Similarity")
ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/cosine_sim_by_year.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved cosine_sim_by_year.csv and .png")

## Part 6 — PyTorch Bag-of-Words Classifier

### Concept: Bag-of-Words (BoW) Text Classification

A **Bag-of-Words** model collapses a document into a vector of term counts
(or TF-IDF scores), discarding word order. The input to the neural network is:

$$\mathbf{x} \in \mathbb{R}^V \quad \text{where } V = \text{vocabulary size}$$

The classifier is a single linear layer:
$$\hat{y} = \text{softmax}(W\mathbf{x} + b) \quad W \in \mathbb{R}^{C \times V}$$

where $C=2$ (NVDA vs AMD).

**Why is this a meaningful baseline?**
Even without understanding syntax or semantics, a BoW model can learn that
certain terms (e.g., "gaming", "data center", "EPYC") are company-specific.

In [ ]:
# ── Cell 9: Prepare train/test split ──────────────────────────────────────────
labels = (chunks_df["ticker"] == "NVDA").astype(int).values   # 1=NVDA, 0=AMD
X = tfidf_matrix.toarray().astype(np.float32)
y = labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

X_train_t = torch.tensor(X_train)
X_test_t  = torch.tensor(X_test)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

print(f"Train: {X_train_t.shape}, Test: {X_test_t.shape}")

In [ ]:
# ── Cell 10: Define and train BoW classifier ──────────────────────────────────
class BoWClassifier(nn.Module):
    def __init__(self, vocab_size: int, n_classes: int = 2):
        super().__init__()
        self.fc = nn.Linear(vocab_size, n_classes)

    def forward(self, x):
        return self.fc(x)

VOCAB_SIZE = X_train_t.shape[1]
model     = BoWClassifier(VOCAB_SIZE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 20
BATCH  = 256
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    perm   = torch.randperm(len(X_train_t))
    ep_loss = 0.0
    for i in range(0, len(X_train_t), BATCH):
        idx    = perm[i:i+BATCH]
        xb, yb = X_train_t[idx], y_train_t[idx]
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item() * len(xb)
    loss_history.append(ep_loss / len(X_train_t))
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}  Loss: {loss_history[-1]:.4f}")

torch.save(model.state_dict(), f"{ARTIFACTS}/bow_classifier_weights.pt")
print("Saved bow_classifier_weights.pt")

In [ ]:
# ── Cell 11: Evaluate — confusion matrix & classification report ───────────────
model.eval()
with torch.no_grad():
    preds = model(X_test_t).argmax(dim=1).numpy()

print(classification_report(y_test, preds, target_names=["AMD", "NVDA"]))

cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["AMD", "NVDA"], yticklabels=["AMD", "NVDA"], ax=ax)
ax.set_title("Confusion Matrix — BoW Classifier")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/bow_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved bow_confusion_matrix.png")

In [ ]:
# ── Cell 12: Loss curve ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3))
sns.lineplot(x=range(1, EPOCHS+1), y=loss_history, marker="o", ax=ax)
ax.set_title("BoW Classifier Training Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-Entropy Loss")
plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/bow_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## Part 7 — Cosine Similarity Baseline

Before accepting the neural classifier result, compare it to a simple baseline:
**assign each test chunk the label of whichever company centroid it is closest to**.

This is a *zero-parameter* baseline — no training, just geometry.

In [ ]:
# ── Cell 13: Cosine similarity baseline ───────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Compute per-class centroids from training data
nvda_centroid = X_train[y_train == 1].mean(axis=0, keepdims=True)
amd_centroid  = X_train[y_train == 0].mean(axis=0, keepdims=True)

sims_nvda = cos_sim(X_test, nvda_centroid).flatten()
sims_amd  = cos_sim(X_test, amd_centroid).flatten()
baseline_preds = (sims_nvda > sims_amd).astype(int)

baseline_acc  = accuracy_score(y_test, baseline_preds)
classifier_acc = accuracy_score(y_test, preds)

summary = pd.DataFrame({
    "Method":   ["Cosine Baseline", "BoW Classifier"],
    "Accuracy": [baseline_acc, classifier_acc],
})
summary["Accuracy %"] = (summary["Accuracy"] * 100).round(2)
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(data=summary, x="Method", y="Accuracy", palette=["#6baed6","#2171b5"], ax=ax)
ax.set_ylim(0, 1.1)
ax.set_title("Classification Accuracy: Baseline vs BoW Classifier")
ax.set_ylabel("Accuracy (0–1)")
for bar, val in zip(ax.patches, summary["Accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}",
            ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Assignment Questions — Model Answers

### Q1 — Which 10-K section (Item 1A or Item 7) produces higher classification accuracy, and why?

**Model Answer:**
Item 1A (Risk Factors) typically yields higher accuracy because it contains
highly company-specific language about competitive risks, product lines, and
market positions (e.g., NVIDIA's mentions of "data center", "CUDA", "H100"
versus AMD's mentions of "EPYC", "Radeon", "Xilinx"). Item 7 (MD&A) uses more
generic financial vocabulary ("revenue", "gross margin", "operating expenses")
that is shared across companies in the same sector, reducing discriminative signal.

### Q2 — Does the BoW classifier outperform the cosine similarity baseline?

**Model Answer:**
Yes. The BoW classifier learns a *discriminative* decision boundary —
it is trained to minimize misclassification — while the cosine baseline is
*generative* (closest centroid). The classifier can weight rare but highly
informative bigrams more aggressively. Expect the classifier to outperform
the baseline by 5–15 percentage points depending on corpus size.

### Q3 — Business Recommendation

**Model Answer:**
The high classification accuracy (typically >85%) shows that NVIDIA and AMD
use **systematically different language** in their filings despite operating in
the same NAICS-3344 vertical. NVIDIA's 10-Ks increasingly emphasize AI
infrastructure and hyperscaler partnerships, while AMD's emphasize CPU-GPU
integration and data-center CPU momentum (EPYC). An analyst monitoring these
filings with an automated NLP pipeline could detect strategic shifts earlier
than traditional earnings-call analysis — for example, a rising cosine
similarity between the two companies' filings would signal that AMD is
directly competing in NVIDIA's core markets.

---
### Artifact Summary
| File | Description |
|---|---|
| `company_context.csv` | Yahoo Finance market data |
| `corpus.csv` | Raw 10-K text per company/year |
| `chunks.csv` | 400-word chunks (used by A2, A3, A4) |
| `tfidf_matrix.csv` | First 1000 TF-IDF columns |
| `vocab.json` | Full 5000-token vocabulary |
| `bow_classifier_weights.pt` | Trained BoW model weights |
| `top_tfidf_terms.png` | Top-20 terms per company |
| `cosine_sim_by_year.png` | Similarity trend 2020–2024 |
| `bow_confusion_matrix.png` | Classifier confusion matrix |
| `accuracy_comparison.png` | Baseline vs classifier bar chart |